In [0]:
from pyspark.sql.functions import (
    col, to_timestamp, to_date, when, lit, sum as _sum, first, coalesce, date_sub 
)

df_news_silver = spark.table("crypto_sentiment_silver_news")

df_sentiment_daily = df_news_silver.groupBy("symbol", "name", "news_date").agg(
    _sum("sentiment_score").alias("total_daily_score"),
    first("news_headline").alias("top_headline") # Top headline for context
)


display(df_sentiment_daily)

symbol,name,news_date,total_daily_score,top_headline
BTC,BITCOIN,2026-02-03,3,"Bitcoinwell.com, a bitcoin-only platform, promotes Bitcoin as a tool for independence in discussions surrounding government policy."
DOGE,DOGECOIN,2026-02-03,1,"Analyst Trader Tardigrade suggests Dogecoin (DOGE) may be preparing for another parabolic rally, citing historical Price Momentum Oscillator patterns."
ETH,ETHEREUM,2026-02-03,1,"Ethereum is among the major blockchain networks chosen for USDC funding distribution, signaling continued relevance and use."
HYPE,HYPERLIQUID,2026-02-03,1,Hyperliquid's expansion into prediction markets and options is anticipated to fuel the HYPE token's third leg of recovery.
SHIB,SHIBA INU,2026-02-03,-1,"An analyst flagged a brutal downside scenario for Shiba Inu (SHIB), where a breakdown could lead to an 81% price drop."
SOL,SOLANA,2026-02-03,1,"Solana is among the major blockchain networks chosen for USDC funding distribution, signaling continued relevance and use."
USDC,USD COIN,2026-02-03,1,"YC's Nemil Dalal announced that funding will be distributed in USDC across major blockchain networks, indicating increased utility and adoption for the stablecoin."
XLM,STELLAR,2026-02-03,1,"Rails is leveraging Stellar-based smart contract vaults and on-chain proofs to make high-speed perpetuals more attractive to institutions, suggesting increased institutional adoption for Stellar."


In [0]:
df_market_silver = spark.table("crypto_sentiment_silver_price")

df_gold = df_market_silver.join(
    df_sentiment_daily,
    (df_market_silver.symbol == df_sentiment_daily.symbol) & 
    (date_sub(df_market_silver.market_date, 1) == df_sentiment_daily.news_date),
    how="left"
).drop(df_sentiment_daily.symbol)

df_gold = df_gold.withColumn("total_daily_score", coalesce(col("total_daily_score"), lit(0)))

In [0]:
df_gold = df_gold.withColumn(
    "trade_signal",
    # SCENARIO 1: Strong Buy (Price Up + Positive News + High Vol)
    when(
        (col("percent_change_24h") > 0) & 
        (col("total_daily_score") > 0) & 
        (col("volume_24h") > 1000000), 
        "STRONG BUY"
    )
    # SCENARIO 2: Buy the Dip (Price Down + Positive News)
    .when(
        (col("percent_change_24h") < -2) & 
        (col("total_daily_score") > 0), 
        "VALUE BUY (Dip)"
    )
    # SCENARIO 3: Bearish Divergence (Price Up + Negative News) -> Watch out!
    .when(
        (col("percent_change_24h") > 0) & 
        (col("total_daily_score") < 0), 
        "WARNING (Fakeout)"
    )
    # SCENARIO 4: Panic Sell (Price Down + Negative News)
    .when(
        (col("percent_change_24h") < 0) & 
        (col("total_daily_score") < 0), 
        "STRONG SELL"
    )
    .otherwise("HOLD / NEUTRAL")
).display()

symbol,price,cmc_rank,volume_24h,percent_change_1h,percent_change_24h,percent_change_7d,market_timestamp,ingested_at,market_date,risk_category,name,news_date,total_daily_score,top_headline,trade_signal
XLM,0.1697,18,181632390,-1.17,-2.78,-19.3,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z,2026-02-04,Mid Cap,STELLAR,2026-02-03,1,"Rails is leveraging Stellar-based smart contract vaults and on-chain proofs to make high-speed perpetuals more attractive to institutions, suggesting increased institutional adoption for Stellar.",VALUE BUY (Dip)
BNB,735.4924,4,2147483647,-0.68,-4.05,-18.42,2026-02-04T15:06:00Z,2026-02-04T15:06:57Z,2026-02-04,Blue Chip,null,null,0,null,HOLD / NEUTRAL
XRP,1.5435,5,2147483647,-1.44,-2.6,-19.61,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z,2026-02-04,Blue Chip,null,null,0,null,HOLD / NEUTRAL
LINK,9.2558,16,1061111421,-1.77,-2.92,-21.79,2026-02-04T15:06:00Z,2026-02-04T15:06:57Z,2026-02-04,Mid Cap,null,null,0,null,HOLD / NEUTRAL
USDT,0.9983,3,2147483647,-0.02,-0.09,-0.04,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z,2026-02-04,Blue Chip,null,null,0,null,HOLD / NEUTRAL
DOGE,0.104,9,1968228130,-1.7,-1.85,-16.98,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z,2026-02-04,Blue Chip,DOGECOIN,2026-02-03,1,"Analyst Trader Tardigrade suggests Dogecoin (DOGE) may be preparing for another parabolic rally, citing historical Price Momentum Oscillator patterns.",HOLD / NEUTRAL
CMC20,152.9817,8853,4311513,-1.05,-3.78,-18.8,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z,2026-02-04,Small Cap,null,null,0,null,HOLD / NEUTRAL
DOT,1.4524,32,211072388,-1.59,-3.06,-21.77,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z,2026-02-04,Mid Cap,null,null,0,null,HOLD / NEUTRAL
ADA,0.2872,11,817583525,-1.59,-2.55,-20.0,2026-02-04T15:06:00Z,2026-02-04T15:06:57Z,2026-02-04,Mid Cap,null,null,0,null,HOLD / NEUTRAL
ETH,2169.8794,2,2147483647,-1.23,-4.34,-28.03,2026-02-04T15:05:00Z,2026-02-04T15:06:57Z,2026-02-04,Blue Chip,ETHEREUM,2026-02-03,1,"Ethereum is among the major blockchain networks chosen for USDC funding distribution, signaling continued relevance and use.",VALUE BUY (Dip)
